In [ ]:
df = spark.table('delta_tableName').limit(0) 

In [ ]:
config_row = spark.sql(f"""
    SELECT connection_string, username, password, driver
    FROM connection_db_config      
    WHERE is_active = true
    AND connection_name = 'your_connection_name'
    """).first()
if not config_row:
    print("Connection config not found")
    stop_interpreter()
    
connection_string = config_row['connection_string']
username = config_row['username'] 
password = config_row['password']
driver = config_row['driver']

df.write \
    .format("jdbc") \
    .option("url", connection_string) \
    .option("dbtable", f"delta_tableName") \
    .option("user", username) \
    .option("password", password) \
    .option("driver", driver) \
    .mode("append") \
    .save()

In [ ]:
df = spark.sql(f"""
SELECT * FROM delta_tableName
""")
df.count()

In [ ]:
table_name = "delta_tableName"
 
properties = {
  "user": username,
  "password": password,
  "driver": driver,
  "batchsize": "10000"
}

# For initial load, use "overwrite" mode. For subsequent upserts, you can change it to "append" or implement a more complex upsert logic as needed.
df.write.option("truncate", "true").mode("overwrite").jdbc(url=connection_string, table=table_name, properties=properties)